# Local AI Sales Intelligence

A complete source walkthrough in ten chapters. Each chapter contains explanations, full file-writing code cells, runnable fictional-data examples and a verification checkpoint.

Start Jupyter from this project using its virtual environment. Read `README.md` first. The delivered source files are already present. Source cells reconstruct a separate copy in `notebook_project/`; demonstrations import the supplied, tested package.

All demonstrations use clearly fictional transactions. Model and n8n calls are opt-in. `Run All` does not install system software, download models, publish workflows, or start services. To review one chapter at a time, use the individual notebooks and pause at each checkpoint.

The original prompt's ten topics are retained. Basic deterministic preparation comes before optional local models, and the complete web API is assembled after its modules exist.


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the supplied project or its notebooks directory."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Source-writing cells reconstruct a separate copy and preserve the supplied project.
BUILD_ROOT = ROOT / "notebook_project"
BUILD_ROOT.mkdir(exist_ok=True)
for folder in ["sales_intelligence", "scripts", "tests", "workflows", "frontend/src", "docs"]:
    (BUILD_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_LOCAL_MODELS = False  # opt in only after installing local Ollama models
RUN_N8N = False          # opt in only after configuring/publishing the local workflows
print("Python:", sys.version.split()[0])
print("Source reconstruction:", BUILD_ROOT)


## Chapter 8 · Local RAG and text analysis

Extract bounded TXT/MD, text PDF or DOCX content, chunk it, embed through local Ollama and index vectors using FAISS. Document/model hashes prevent redundant embedding. Use deterministic TF-IDF and NMF for text exploration.

### Code cell · `sales_intelligence/rag.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/rag.py'}"
import hashlib
import io
import json
from pathlib import Path

import faiss
import numpy as np
from docx import Document
from pypdf import PdfReader

from .ingestion import atomic_json, check_archive


def extract_text(filename, body):
    extension = Path(filename.replace("\\", "/")).suffix.lower()
    if not body or len(body) > 10 * 1024 * 1024:
        raise ValueError("Documents must be nonempty and at most 10 MB.")
    if extension in {".txt", ".md"}:
        text = body.decode("utf-8-sig")
    elif extension == ".pdf":
        reader = PdfReader(io.BytesIO(body))
        if reader.is_encrypted or len(reader.pages) > 100:
            raise ValueError("Encrypted PDFs and PDFs exceeding 100 pages are not supported.")
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif extension == ".docx":
        check_archive(body, maximum_uncompressed=30_000_000)
        document = Document(io.BytesIO(body))
        text = "\n".join(
            [p.text for p in document.paragraphs]
            + [" | ".join(cell.text for cell in row.cells) for table in document.tables for row in table.rows]
        )
    else:
        raise ValueError("Supported documents: UTF-8 TXT/MD, text-based PDF, DOCX. OCR is not included.")
    if not text.strip() or len(text) > 500_000:
        raise ValueError("No extractable text or document exceeds 500,000 characters.")
    return text


def add_document(store, dataset_id, filename, body, model):
    directory = store.directory(dataset_id) / "documents"
    directory.mkdir(exist_ok=True)
    text = extract_text(filename, body)
    model_digest = model.verify_downloaded(model.settings.embedding_model)
    content_hash = hashlib.sha256(body).hexdigest()
    identity = hashlib.sha256((content_hash + model_digest + "chunk-v1-800-100").encode()).hexdigest()
    manifest = directory / f"{identity}.json"
    if manifest.exists():
        return {
            "document_id": identity,
            "cached": True,
            "chunks": len(json.loads(manifest.read_text())["chunks"]),
        }
    chunks = [
        text[index : index + 800] for index in range(0, len(text), 700) if text[index : index + 800].strip()
    ]
    vectors = np.array(model.embed(chunks), dtype="float32")
    if vectors.ndim != 2 or len(vectors) != len(chunks) or not np.isfinite(vectors).all():
        raise ValueError("Invalid embedding vectors.")
    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    faiss.write_index(index, str(directory / f"{identity}.faiss"))
    (directory / f"{identity}{Path(filename).suffix.lower()}").write_bytes(body)
    atomic_json(
        manifest,
        {
            "id": identity,
            "source": Path(filename.replace("\\", "/")).name[:200],
            "content_hash": content_hash,
            "embedding_digest": model_digest,
            "chunks": chunks,
        },
    )
    store.audit(
        dataset_id, "document_indexed", {"id": identity, "content_hash": content_hash, "chunks": len(chunks)}
    )
    return {"document_id": identity, "cached": False, "chunks": len(chunks)}


def retrieve(store, dataset_id, question, model, limit=4):
    directory = store.directory(dataset_id) / "documents"
    if not directory.exists() or not list(directory.glob("*.json")):
        return []
    digest = model.verify_downloaded(model.settings.embedding_model)
    vector = np.array(model.embed([question]), dtype="float32")
    faiss.normalize_L2(vector)
    matches = []
    for path in directory.glob("*.json"):
        manifest = json.loads(path.read_text())
        if manifest["embedding_digest"] != digest:
            continue
        index = faiss.read_index(str(path.with_suffix(".faiss")))
        if vector.shape[1] != index.d:
            continue
        scores, indices = index.search(vector, min(limit, index.ntotal))
        for score, number in zip(scores[0], indices[0]):
            if number >= 0:
                matches.append(
                    {
                        "source_id": f"{manifest['id']}:{number}",
                        "source": manifest["source"],
                        "chunk": int(number),
                        "similarity": float(score),
                        "text": manifest["chunks"][number],
                        "type": "organizational_document",
                    }
                )
    return sorted(matches, key=lambda item: item["similarity"], reverse=True)[:limit]


def evaluate_retrieval(cases, search):
    if not cases:
        raise ValueError("Supply questions with human-labeled relevant source IDs.")
    scores = []
    for case in cases:
        relevant = set(case["relevant_source_ids"])
        found = {result["source_id"] for result in search(case["question"])}
        scores.append(
            {
                "question": case["question"],
                "precision": len(found & relevant) / max(len(found), 1),
                "recall": len(found & relevant) / max(len(relevant), 1),
            }
        )
    return scores


### Code cell · `sales_intelligence/nlp.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/nlp.py'}"
import re

from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline

from .preparation import load_clean
from .schemas import TextClassifier


def text_analysis(store, dataset_id):
    frame, data = load_clean(store, dataset_id)
    if "text" not in frame or frame.text.dropna().empty:
        return {"status": "skipped", "reason": "No nonempty mapped text column exists."}
    texts = frame.text.dropna().astype(str).map(lambda value: value[:4000]).tolist()
    texts = [re.sub(r"\s+", " ", text.replace("ي", "ی").replace("ك", "ک")) for text in texts]
    try:
        vectorizer = TfidfVectorizer(
            max_features=2000,
            max_df=1.0,
            min_df=1,
            stop_words=["the", "and", "is", "of", "to", "از", "به", "در", "و", "این", "که", "را"],
        )
        matrix = vectorizer.fit_transform(texts)
    except ValueError:
        return {"status": "skipped", "reason": "No usable text tokens remain."}
    terms = vectorizer.get_feature_names_out()
    weights = matrix.mean(axis=0).A1
    keywords = [
        {"term": terms[index], "weight": float(weights[index])} for index in weights.argsort()[-20:][::-1]
    ]
    topics = []
    if len(texts) >= 20 and matrix.shape[1] >= 10:
        model = NMF(n_components=min(4, matrix.shape[1] - 1), init="nndsvda", random_state=42, max_iter=300)
        model.fit(matrix)
        topics = [
            [terms[index] for index in component.argsort()[-8:][::-1]] for component in model.components_
        ]
    return {
        "dataset_version": data["version"],
        "status": "completed",
        "documents": len(texts),
        "keywords": keywords,
        "topics": topics,
        "sentiment": {
            "status": "not_estimated",
            "reason": "No evaluated Persian/domain-specific sentiment classifier has been supplied.",
        },
        "complaint_classification": {
            "status": "not_estimated",
            "reason": "Requires a defined complaint taxonomy and labeled evaluation set.",
        },
        "limitation": "TF-IDF keywords and NMF topics are exploratory. No sentiment or complaint labels are fabricated.",
    }


def evaluate_text_classifier(store, dataset_id, request: TextClassifier):
    """Train sentiment/complaint classification only from reviewed, existing labels."""
    if not request.labeling_reviewed:
        raise ValueError("A person must review the label taxonomy and its quality.")
    frame, data = load_clean(store, dataset_id)
    if "text" not in frame or request.label_column not in frame:
        raise ValueError("Map text and supply an existing sentiment/complaint label column.")
    labeled = frame.dropna(subset=["text", request.label_column, "transaction_date"]).copy()
    labeled["text"] = labeled.text.astype(str).str.strip()
    labeled[request.label_column] = labeled[request.label_column].astype(str)
    if len(set(request.permitted_labels)) != len(request.permitted_labels):
        raise ValueError("Label names must be unique.")
    if not set(labeled[request.label_column]) <= set(request.permitted_labels):
        raise ValueError("A label is outside the confirmed taxonomy.")
    conflicts = labeled.groupby("text")[request.label_column].nunique()
    if (conflicts > 1).any():
        raise ValueError("Identical text has conflicting labels. Review the labels first.")
    # De-duplicate the evaluation view only; the cleaned dataset is not changed.
    labeled = labeled.sort_values("transaction_date").drop_duplicates(subset=["text"], keep="first")
    if len(labeled) < 100:
        raise ValueError("Need at least 100 distinct, labeled texts with dates for evaluation.")
    cutoff = labeled.transaction_date.quantile(0.8)
    train = labeled.transaction_date < cutoff
    test = ~train
    if train.sum() < 50 or test.sum() < 20:
        raise ValueError("Too few distinct dates for text-classifier evaluation.")
    y = labeled[request.label_column]
    counts = y.loc[train].value_counts()
    if counts.min() < 10 or len(counts) < 2 or not set(y.loc[test]) <= set(counts.index):
        raise ValueError("Need two training labels with 10 examples each and no unseen holdout labels.")
    pipeline = make_pipeline(
        TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True),
        LogisticRegression(class_weight="balanced", max_iter=1000),
    )
    pipeline.fit(labeled.loc[train, "text"], y.loc[train])
    predicted = pipeline.predict(labeled.loc[test, "text"])
    baseline = [counts.idxmax()] * int(test.sum())
    labels = sorted(counts.index)
    return {
        "purpose": request.purpose,
        "dataset_version": data["version"],
        "label_definition": request.label_definition,
        "train_rows": int(train.sum()),
        "test_rows": int(test.sum()),
        "evaluation_deduplicated_texts": True,
        "holdout_start": str(cutoff),
        "labels": labels,
        "f1_macro": float(f1_score(y.loc[test], predicted, average="macro", zero_division=0)),
        "baseline_f1_macro": float(f1_score(y.loc[test], baseline, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(y.loc[test], predicted, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y.loc[test], predicted, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y.loc[test], predicted, labels=labels).tolist(),
        "limitation": "Evaluation only. Requires domain/language review before deployment; labels are never generated by this code.",
    }


### Run the chapter example

In [ ]:
if "demo_store" not in globals():
    from scripts.notebook_demo import make_demo
    demo_store, demo_settings, demo_id, sample_frame = make_demo(ROOT)
print("Fictional dataset:", demo_id)
from sales_intelligence.nlp import text_analysis
from sales_intelligence.rag import add_document, retrieve
display(text_analysis(demo_store, demo_id))
if RUN_LOCAL_MODELS:
    from sales_intelligence.config import Settings
    from sales_intelligence.llm import LocalModel
    local_model = LocalModel(Settings().prepare())
    policy = (ROOT / "sample_data/company_policy.txt").read_bytes()
    display(add_document(demo_store, demo_id, "fictional_policy.txt", policy, local_model))
    display(retrieve(demo_store, demo_id, "What approval is required for product bundles?", local_model))
else:
    print("Live embedding/retrieval not run. Tests exercise FAISS indexing with explicit fixture embeddings.")


### Review checkpoint

Sentiment and complaint classification are reported as not estimated without reviewed labels. evaluate_text_classifier trains and evaluates the supplied taxonomy with TF-IDF, logistic regression and a chronological holdout; it never invents labels. NMF topics are exploratory. Scanned PDFs need OCR outside this implementation. Retrieved documents are not treated as verified transaction facts.

Review this result before moving to the next chapter.